# Telco Customer Churn Analysis

**Dataset:** IBM Telco Customer Churn — 7,043 customers, 21 features  
**Goal:** Identify drivers of churn and build models to predict at-risk customers.

---
## Contents
1. Data Loading & Overview
2. Exploratory Data Analysis (EDA)
3. Feature Engineering & Preprocessing
4. Model Training & Evaluation
5. Feature Importance
6. Business Insights & Recommendations

In [ ]:
import warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
os.makedirs('outputs', exist_ok=True)

## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)
df.drop(columns=['customerID'], inplace=True)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
print(df.dtypes)
print('\nMissing values:', df.isnull().sum().sum())
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
# Churn distribution
counts = df['Churn'].value_counts()
labels = ['No Churn', 'Churn']
colors = ['#4C72B0', '#DD8452']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].pie(counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Overall Churn Distribution')
axes[1].bar(labels, counts.values, color=colors, edgecolor='white')
axes[1].set_ylabel('Customers')
axes[1].set_title('Churn Count')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/01_churn_distribution.png', dpi=150)
plt.show()
print(f'Overall churn rate: {df["Churn"].mean():.1%}')

In [ ]:
# Numeric feature distributions by churn
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    for label, color in [(0, '#4C72B0'), (1, '#DD8452')]:
        ax.hist(df[df['Churn'] == label][col], bins=30, alpha=0.6, color=color,
                label='No Churn' if label == 0 else 'Churn', edgecolor='white')
    ax.set_title(col)
    ax.legend()
plt.suptitle('Numeric Feature Distributions by Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/02_numeric_distributions.png', dpi=150)
plt.show()

In [ ]:
# Categorical churn rates
cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'InternetService', 'Contract',
            'PaperlessBilling', 'PaymentMethod']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.flatten(), cat_cols):
    rate = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    sns.barplot(x=rate.index, y=rate.values, ax=ax, palette='muted')
    ax.set_title(f'Churn Rate by {col}')
    ax.set_ylabel('Churn Rate')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1%}',
                    (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom', fontsize=8)
plt.suptitle('Churn Rate by Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/03_categorical_churn_rates.png', dpi=150)
plt.show()

## 3. Feature Engineering & Preprocessing

In [ ]:
df_enc = df.copy()
le = LabelEncoder()
binary_cols = [c for c in df_enc.select_dtypes('object').columns
               if df_enc[c].nunique() == 2]
for col in binary_cols:
    df_enc[col] = le.fit_transform(df_enc[col])

df_enc = pd.get_dummies(df_enc, drop_first=True)
print(f'Encoded shape: {df_enc.shape}')

# Correlation with Churn
corr = df_enc.corr()[['Churn']].drop('Churn').sort_values('Churn', ascending=False)
fig, ax = plt.subplots(figsize=(6, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation with Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/04_correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
X = df_enc.drop(columns=['Churn'])
y = df_enc['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Address class imbalance with SMOTE
sm = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print(f'After SMOTE — train size: {X_train_res.shape[0]}, churn ratio: {y_train_res.mean():.1%}')

scaler = StandardScaler().fit(X_train_res)

## 4. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'XGBoost':             XGBClassifier(n_estimators=200, use_label_encoder=False,
                                          eval_metric='logloss', random_state=RANDOM_STATE),
}

X_train_sc = scaler.transform(X_train_res)
X_test_sc  = scaler.transform(X_test)
results = []

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, model) in zip(axes.flatten(), models.items()):
    X_tr = X_train_sc if name == 'Logistic Regression' else X_train_res
    X_te = X_test_sc  if name == 'Logistic Regression' else X_test

    model.fit(X_tr, y_train_res)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    auc    = roc_auc_score(y_test, y_proba)
    report = classification_report(y_test, y_pred, output_dict=True)
    results.append({
        'Model':     name,
        'ROC-AUC':   round(auc, 4),
        'Precision': round(report['1']['precision'], 4),
        'Recall':    round(report['1']['recall'], 4),
        'F1-Score':  round(report['1']['f1-score'], 4),
        'Accuracy':  round(report['accuracy'], 4),
    })

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'AUC = {auc:.3f}', lw=2)
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_title(name)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right')

plt.suptitle('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/05_roc_curves.png', dpi=150)
plt.show()

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df.to_csv('outputs/model_results.csv', index=False)
results_df

In [ ]:
# Confusion matrix for XGBoost (best performer)
xgb = models['XGBoost']
y_pred_xgb = xgb.predict(X_test)
cm = confusion_matrix(y_test, y_pred_xgb)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm,
                       display_labels=['No Churn', 'Churn']).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/07_confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(y_test, y_pred_xgb, target_names=['No Churn', 'Churn']))

## 5. Feature Importance

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
rf.fit(X_train_res, y_train_res)
importance = pd.Series(rf.feature_importances_, index=X_train_res.columns)
top20 = importance.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
top20.plot(kind='barh', color='#4C72B0', ax=ax)
ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('outputs/06_feature_importance.png', dpi=150)
plt.show()

## 6. Business Insights & Recommendations

In [ ]:
insights = {
    'Contract type': df.groupby('Contract')['Churn'].mean().to_dict(),
    'Internet service': df.groupby('InternetService')['Churn'].mean().to_dict(),
    'Senior citizen': df.groupby('SeniorCitizen')['Churn'].mean().to_dict(),
}

for k, v in insights.items():
    print(f'\nChurn rate by {k}:')
    for kk, vv in v.items():
        print(f'  {kk}: {vv:.1%}')

q1, q3 = df['tenure'].quantile([0.25, 0.75])
print(f'\nChurn — low tenure (≤{q1:.0f} mo): {df[df["tenure"]<=q1]["Churn"].mean():.1%}')
print(f'Churn — high tenure (≥{q3:.0f} mo): {df[df["tenure"]>=q3]["Churn"].mean():.1%}')

### Key Takeaways

| Finding | Churn Rate | Recommendation |
|---|---|---|
| Month-to-month contracts | ~43% | Incentivise annual contract upgrades |
| Fiber optic customers | ~42% | Investigate service quality / pricing |
| Tenure < 12 months | ~50%+ | Deploy early-life retention programmes |
| Senior citizens | ~41% | Dedicated support & loyalty offers |
| Electronic check payers | ~45% | Encourage autopay migration |

**Model recommendation:** XGBoost achieves the best ROC-AUC and should be used for scoring the customer base monthly. Prioritise outreach to customers in the top-two churn deciles.